# Galaxy viewer

This notebook simply loads a given galaxy (by target id) and plots both the spectrum and some images. 

The target must be in the dwarfs emission line table (stellar_mass_emline_dwarfs.fits). 

The spectrum is pulled from DESI, and the images from the Legacy Survey


In [ ]:
target_id = 39628390604476594

In [ ]:
import re

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

from sparcl.client import SparclClient

In [ ]:
client = SparclClient()

In [ ]:
dwarf_cat = Table.read("stellar_mass_emline_dwarfs.fits")

# Plotting utilities

In [ ]:
emline_cols = [
    'nev3346', 
    'nev3426',
    'oii3726',
    'oii3729',
    'neiii3869',
    'neiii3967',
    'hepsilon',
    'hei4026', 
    'hdelta',
    'hgamma', 
    'oiii4363',
    'hei4471', 
    'heii4686', 
    'hbeta',
    'oiii4959',
    'oiii5007',
    'nii5755',
    'hei5876', 
    'oi6300', 
    'siii6312',
    'nii6548',
    'halpha',  
    'nii6583',
    'sii6716', 
    'sii6731',  
    'hei7065',
    'ariii7136',
    'hei7281',
    'oii7320', 
    'oii7331',
    'ariii7751',
    'siii9071', 
    'siii9533',
]

In [ ]:
balmer_lines = {
    "halpha": 6565,
    "hbeta": 4861,
    "hgamma": 4340,
    "hdelta": 4102,
    "hepsilon": 3970,
}

In [ ]:
line_labels = {
    "halpha": r"H $\alpha$",
    "hbeta": r"H $\beta$",
    "hgamma": r"H $\gamma$",
    "hdelta": r"H $\delta$",
    "hepsilon": r"H $\epsilon$",
    "nev": "[Ne V]",
    "oii": "[O II]",
    "neiii": "[Ne III]",
    "hei": "[He I]",
    "heii": "[He II]",
    "oiii": "[O III]",
    "nii": "[N II]",
    "oi": "[O I]",
    "siii": "[S III]",
    "sii": "[S II]", 
    "ariii": "[Ar III]",
}

In [ ]:
def retrieve_wavelength(line):
    matches = re.findall(r"\d{4}", line)
    if len(matches) == 1:
        return int(matches[0])

    if line in balmer_lines.keys():
        return balmer_lines[line]


    raise Exception(f"Line not known {line}")

In [ ]:
def retrieve_good_lines(galaxy):
    good_lines = []
    for line in emline_cols:
        if galaxy[line + "_flux"] / galaxy[line + "_fluxerr"] > 3:
            good_lines.append(line)

    return good_lines

In [ ]:
def nicer_label(line):
    if line in line_labels:
        return line_labels[line]
    
    else: 
        λ = retrieve_wavelength(line)
        species = line.replace(str(λ), "")
        species = line_labels[species]
        return species + r" $\lambda$" + str(λ)
        

In [ ]:
def plot_lines(meas, z):
    λs = []
    good_lines = retrieve_good_lines(meas)
    
    for line in emline_cols:
        λ = retrieve_wavelength(line)
        λ = λ * (1 + z)
        λs.append(λ)

        if line in good_lines:
            color = "k"
        else:
            color = "grey"
            
        plt.plot([λ, λ], [-10, -8], color=color)
        offset = line_position_shifts.get(line, (0,0))
        plt.annotate(nicer_label(line), (λ, -10), offset, 
                     va="top", ha="center", rotation=90,
                    xycoords = "data",
                    textcoords="offset fontsize", color=color)


In [ ]:
from astropy.convolution import convolve, Gaussian1DKernel

# Active loop

In [ ]:
idx = np.where(dwarf_cat["targetid"] == target_id)[0][0]

In [ ]:
idx

In [ ]:
target_id_retrieved = int(dwarf_cat["targetid"][idx])

In [ ]:
assert target_id_retrieved == target_id

In [ ]:
inc = ['specid', 'redshift', 'flux', 'wavelength', 'spectype', 'specprimary', 'survey', 'program', 'targetid', 'redshift_warning']
res = client.retrieve_by_specid(specid_list = [target_id], include = inc,
                                dataset_list = ['DESI-DR1'])


In [ ]:
res

In [ ]:
records = res.records

# Select the primary spectrum
spec_primary = np.array([records[jj].specprimary for jj in range(len(records))])

primary_ii = np.where(spec_primary == True)[0][0]



In [ ]:
meas = dwarf_cat[idx]
meas

In [ ]:
z = meas["z"]
z

In [ ]:
emline_cols

In [ ]:
smoothing_kernel = Gaussian1DKernel(5)

In [ ]:
line_position_shifts = {
    "neiii3967": (0, -3),
    "hgamma": (-0.5, 0),
    "oi6300": (-0.5, 0),
    "siii6312": (0.5, 0),
    "nii6548": (-0.5, 0), 
    "nii6583": (0.5, 0),
    "oii7331": (1, 0),
    "sii6716": (0.3, 0),
    "sii6731": (-0.7, 0)

}

In [ ]:
lam_primary = records[primary_ii].wavelength
flam_primary = records[primary_ii].flux

# Plotting this spectrum

plt.figure(figsize = (20, 6))

# Plot the original spectrum in maroon color
plt.plot(lam_primary, flam_primary, color = 'maroon', alpha = 0.5, lw=0.5)

# Over-plotting smoothed spectra in black 
plt.plot(lam_primary, convolve(flam_primary, smoothing_kernel), color = 'k', lw = 2.0)
plot_lines(meas, z)

plt.axhline(0)

plt.xlim(np.min(lam_primary) - 50, np.max(lam_primary)+50)
plt.xlabel(r'$\lambda$ [$\AA$] (observed)')
plt.ylabel(r'$F_{\lambda}$ [$10^{-17} erg\ s^{-1}\ cm^{-2}\ \AA^{-1}$]')
plt.show()



# Find image of galaxy

In [ ]:
from astropy.utils.data import download_file
from astropy.io import fits
from astropy.wcs import WCS
from astropy.stats import mad_std
from astropy.coordinates import SkyCoord
from astropy.nddata.blocks import block_reduce
from astropy import visualization as aviz

import seaborn
import matplotlib

from pyvo.dal import sia

In [ ]:
DEF_ACCESS_URL = "https://datalab.noirlab.edu/sia/ls_dr10"
svc_lss = sia.SIAService(DEF_ACCESS_URL)


In [ ]:
ra = meas["ra"]
dec = meas["dec"]

In [ ]:
ra, dec

In [ ]:
fov = 0.3 # in degrees
img_table = svc_lss.search((ra,dec), (fov/np.cos(dec*np.pi/180), fov), verbosity=2).to_table()

img_filter = img_table["prodtype"] == "image"

img_table = img_table[img_filter]


In [ ]:
img_table

In [ ]:
def download_image(imgTable, row_val):
    '''Take in a row number from the image table, 
    then return an image, access URL, and its WCS information.
    '''
    row = imgTable[row_val]
    url = row['access_url']
    filename = download_file(url, cache=True, show_progress=False, timeout=120)
    hdu = fits.open(filename)[0]
    image = hdu.data
    hdr = hdu.header
    wcs = WCS(hdr)

    return image, url, wcs


In [ ]:
all_images = [download_image(img_table, i) for i in range(len(img_table))]


In [ ]:
img_size = 200

In [ ]:
def plot_circle(x, y, R, **kwargs):
    patch = matplotlib.patches.Arc((x, y), R, R, **kwargs)

    plt.gca().add_artist(patch)

In [ ]:
cmap = seaborn.color_palette("mako", as_cmap=True)


In [ ]:
def show_image(image,
               percl=99, percu=None, is_mask=False,
               figsize=(10, 10),
               dpi = None,
               cmap=cmap, log=False, clip=True,
               clabel=None,
               clim=None,
               show_colorbar=True, show_ticks=True,
               fig=None, ax=None, input_ratio=None):
    """
    Show an image in matplotlib with some basic astronomically-appropriate stretching.

    Parameters
    ----------
    image
        The image to show
    percl : number
        The percentile for the lower edge of the stretch (or both edges if ``percu`` is None)
    percu : number or None
        The percentile for the upper edge of the stretch (or None to use ``percl`` for both)
    figsize : 2-tuple
        The size of the matplotlib figure in inches
    """
    if percu is None:
        percu = percl
        percl = 100 - percl

    if dpi is None:
        dpi = image.shape[0] / figsize[0]

    if (fig is None and ax is not None) or (fig is not None and ax is None):
        raise ValueError('Must provide both "fig" and "ax" '
                         'if you provide one of them')
    elif fig is None and ax is None:
        if figsize is not None:
            # Rescale the fig size to match the image dimensions, roughly
            image_aspect_ratio = image.shape[0] / image.shape[1]
            figsize = (max(figsize) * image_aspect_ratio, max(figsize))

        fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=dpi)


    # To preserve details we should *really* downsample correctly and
    # not rely on matplotlib to do it correctly for us (it won't).

    # So, calculate the size of the figure in pixels, block_reduce to
    # roughly that,and display the block reduced image.

    # Thanks, https://stackoverflow.com/questions/29702424/how-to-get-matplotlib-figure-size

    fig_size_pix = fig.get_size_inches() * fig.dpi

    ratio = (image.shape // fig_size_pix).max()

    if ratio <= 1:
        ratio = 1
    else:
        print("warning, reducing image size by a factor of", ratio,)

    ratio = input_ratio or ratio

    reduced_data = block_reduce(image, ratio)

    if not is_mask:
        # Divide by the square of the ratio to keep the flux the same in the
        # reduced image. We do *not* want to do this for images which are
        # masks, since their values should be zero or one.
         reduced_data = reduced_data / ratio**2

    # Of course, now that we have downsampled, the axis limits are changed to
    # match the smaller image size. Setting the extent will do the trick to
    # change the axis display back to showing the actual extent of the image.
    extent = [0, image.shape[1], 0, image.shape[0]]

    if log:
        stretch = aviz.LogStretch()
    else:
        stretch = aviz.LinearStretch()

    if clim is None:
        clim = aviz.AsymmetricPercentileInterval(percl, percu)
    else:
        clim = aviz.ManualInterval(*clim)

    norm = aviz.ImageNormalize(reduced_data,
                               interval=clim,
                               stretch=stretch, clip=clip)

    if is_mask:
        # The image is a mask in which pixels should be zero or one.
        # block_reduce may have changed some of the values, so reset here.
        reduced_data = reduced_data > 0
        # Set the image scale limits appropriately.
        scale_args = dict(vmin=0, vmax=1)
    else:
        scale_args = dict(norm=norm)

    im = ax.imshow(reduced_data, origin='lower',
                   cmap=cmap, extent=extent, aspect='equal', **scale_args)

    if show_colorbar:
        # I haven't a clue why the fraction and pad arguments below work to make
        # the colorbar the same height as the image, but they do....unless the image
        # is wider than it is tall. Sticking with this for now anyway...
        # Thanks: https://stackoverflow.com/a/26720422/3486425
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=clabel)
        # In case someone in the future wants to improve this:
        # https://joseph-long.com/writing/colorbars/
        # https://stackoverflow.com/a/33505522/3486425
        # https://matplotlib.org/mpl_toolkits/axes_grid/users/overview.html#colorbar-whose-height-or-width-in-sync-with-the-master-axes

    if not show_ticks:
        ax.tick_params(labelbottom=False, labelleft=False, labelright=False, labeltop=False)


In [ ]:
bandpasses = np.unique(img_table["obs_bandpass"])

In [ ]:
all_bandpasses = ["g", "r", "i", "z"]

observed_bandpasses = [bandpass for bandpass in all_bandpasses if bandpass in bandpasses]
observed_bandpasses

In [ ]:
print(ra, dec)

In [ ]:
plt.imshow(all_images[0][0], norm="log")

wcs1 = all_images[0][-1]
x_gal, y_gal = wcs1.world_to_pixel(SkyCoord(ra, dec, unit="deg"))

plt.xlim(x_gal -img_size, x_gal+img_size)
plt.ylim(y_gal-img_size, y_gal+img_size)

In [ ]:
fig = plt.figure(dpi=500, figsize=(6*len(observed_bandpasses), 6))

for i in range(len(observed_bandpasses)):
    bandpass = observed_bandpasses[i]
    
    row = np.where(img_table["obs_bandpass"] == bandpass)[0][0]
    image1, link1, wcs1 = all_images[row]
    
    x_gal, y_gal = wcs1.world_to_pixel(SkyCoord(ra, dec, unit="deg"))
    ax1 = fig.add_subplot(1, len(observed_bandpasses), i+1, projection=wcs1)

    cmin = mad_std((image1))
    cmax = 200*cmin
    im1 = show_image(image1, ax=ax1, fig=fig, show_colorbar=False, percl=95, log=True, clim=(cmin, cmax))
    
    plot_circle(x_gal, y_gal, 50, color="red")
    
    ax1.set(title = bandpass,
           xlim = (x_gal -img_size, x_gal+img_size),
           ylim = (y_gal-img_size, y_gal+img_size))

    ax1.invert_xaxis()


In [ ]:
meas["targetid"]

In [ ]:
target_id

# LSS image retrieval

In [ ]:
from astropy.nddata import CCDData

In [ ]:
def download_lss_image(link):
    filename = download_file(link, cache=True, show_progress=False, timeout=120)
    hdu = fits.open(filename)[0]
    image = hdu.data
    hdr = hdu.header
    wcs = WCS(hdr)

    return CCDData(image, header=hdr, wcs=wcs, unit="nanomaggy")


In [ ]:
from astropy.nddata import block_reduce
def show_colour_image(image,
               figsize=(10, 10),
               dpi = None,
               clabel=None,
               clim=None,
               show_colorbar=True, show_ticks=True,
               fig=None, ax=None, input_ratio=None,
              subplot_kw=dict()):
    """
    Show an image in matplotlib with some basic astronomically-appropriate stretching.

    Parameters
    ----------
    image
        The image to show
    percl : number
        The percentile for the lower edge of the stretch (or both edges if ``percu`` is None)
    percu : number or None
        The percentile for the upper edge of the stretch (or None to use ``percl`` for both)
    figsize : 2-tuple
        The size of the matplotlib figure in inches
    """

    if dpi is None:
        dpi = image.shape[0] / figsize[0]

    if (fig is None and ax is not None) or (fig is not None and ax is None):
        raise ValueError('Must provide both "fig" and "ax" '
                         'if you provide one of them')
    elif fig is None and ax is None:
        if figsize is not None:
            # Rescale the fig size to match the image dimensions, roughly
            image_aspect_ratio = image.shape[0] / image.shape[1]
            figsize = (max(figsize) * image_aspect_ratio, max(figsize))

        fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=dpi, subplot_kw=subplot_kw)


    # To preserve details we should *really* downsample correctly and
    # not rely on matplotlib to do it correctly for us (it won't).

    # So, calculate the size of the figure in pixels, block_reduce to
    # roughly that,and display the block reduced image.

    # Thanks, https://stackoverflow.com/questions/29702424/how-to-get-matplotlib-figure-size

    fig_size_pix = fig.get_size_inches() * fig.dpi

    ratio = (image.shape[0:2] // fig_size_pix).max()

    if ratio <= 1:
        ratio = 1
    else:
        print("downsampling image by", ratio)

    ratio = input_ratio or ratio

    reduced_data = block_reduce(image, ratio)

    reduced_data = (reduced_data - clim[0]) / (clim[1] - clim[0])

    reduced_data[reduced_data > 1] = 1
    reduced_data[reduced_data < 0] = 0
    
    # Divide by the square of the ratio to keep the flux the same in the
    # reduced image. We do *not* want to do this for images which are
    # masks, since their values should be zero or one.
    # reduced_data = reduced_data / ratio**2

    # Of course, now that we have downsampled, the axis limits are changed to
    # match the smaller image size. Setting the extent will do the trick to
    # change the axis display back to showing the actual extent of the image.
    extent = [0, image.shape[1], 0, image.shape[0]]


    im = ax.imshow(reduced_data, origin='lower',
                   extent=extent, aspect='equal')


    if not show_ticks:
        ax.tick_params(labelbottom=False, labelleft=False, labelright=False, labeltop=False)


In [ ]:
link = f"https://www.legacysurvey.org/viewer/cutout.fits?ra={ra}&dec={dec}&layer=ls-dr11&pixscale=0.262&bands=giz"

In [ ]:
img = download_lss_image(link)

In [ ]:
img.header

In [ ]:
def get_img_data(img):
    img_data = np.moveaxis(img.data, 0, -1)[:, :, ::-1]

    return img_data

In [ ]:
img_data = get_img_data(img)

In [ ]:
def color_clip_image(img, vmin=0, vmax=None, norm=np.arcsinh, scale=1):
    img_clipped = img.copy()
    I = np.sum(img, axis=2) / 3
    if vmax is None:
        h = np.quantile(I[np.isfinite(I)], 0.99)
    else:
        h = vmax
    l = vmin

    def f(I):
        x = (norm(I/scale) - norm(l/scale)) / (norm(h/scale) - norm(l/scale))
        return x
    for i in range(3):
        img_clipped[:, :, i] *= f(I) / I

    return img_clipped

In [ ]:
def ra_dec_axis():
    plt.xlabel(r"Right Ascension")
    plt.ylabel(r"Declination")

In [ ]:
zeropoint = [-0.00, 0.000, 0.00]

In [ ]:
show_colour_image(color_clip_image(img_data + zeropoint, scale=1e-2, vmax=1e1), 
                  clim=(0, 1), figsize=(3, 3), subplot_kw=dict(projection=wcs[0]))
ra_dec_axis()

In [ ]:

img